In [ ]:
import sys
import os
import numpy as np
import warnings
import pandas as pd 
import matplotlib.pyplot as plt

warnings.simplefilter(action='ignore', category=FutureWarning)
%matplotlib inline
current_dir = os.getcwd()

parent_dir = os.path.dirname(current_dir)
parent_dir = os.path.dirname(parent_dir)

sys.path.append(parent_dir)

from model_build_new import InfiltrationModel


Caldwell et all 2019 published and released TxSON network meteorological and soil moisure measurement from 40 different stations. Here is the location of the network (https://doi.org/10.2136/vzj2019.04.0034)

<img src="./figures/net1.png" width="300"> | <img src="./figures/net.jpg" width="300">


6 out of 40 has meteorological stations which measuring the radiation, wind speed etc with hourly temporal resolution. 
### ET Calculation and Partitioning Logic
The potential evapotranspiration ($ET_0$) is calculated using FAO-56 Penman-Monteith equation (solar radiation, air temperature, relative humidity, and wind speed to estimate the total evaporative demand of the atmosphere). The total $ET_0$ is then partitioned into Bare Soil Evaporation ($E_0$) and Canopy Transpiration ($T_0$) using the Leaf Area Index (LAI) by $SCF$, based on Beer’s Law:$$SCF = 1 - e^{-k \cdot LAI}$$Where $k$ is the light extinction coefficient (set to 0.56). The partitioning follows these physical principles:Transpiration ($T_0$): Calculated as $ET_0 \times SCF$, representing the energy intercepted by the leaves.Evaporation ($E_0$): Calculated as $ET_0 \times (1 - SCF)$, representing the energy that reaches the soil surface. MODIS 8 day 500m resolution LAI was used as vegetation data source.

<img src="./figures/meteorology_10_4.png" width="500">


### Definitionn of Soil Hydraulic Parameters

Van Genuchten soil hydraulic properties were derived using the Rosetta3 model (10.1016/j.jhydrol.2017.01.004), based on field-measured texture (sand, silt, clay) and bulk density data from a 10 cm depth.

In [ ]:
meta_dir = os.path.join(os.getcwd(),'hydrus_inputs','soil_properties_rosetta3.csv')
df_soil = pd.read_csv(meta_dir,index_col=0)
df_soil[['theta_r','theta_s','alpha','npar','ksat','L']]

In [ ]:
soil_props = {"a": [df_soil.loc['10_4','alpha'].item()],
              "tr": [df_soil.loc['10_4','theta_r'].item()],
              "ths": [df_soil.loc['10_4','theta_s'].item()],
              "ks": [df_soil.loc['10_4','ksat'].item() / 1440], # cm/day to cm/min conversion
              "n": [df_soil.loc['10_4','npar'].item()],
              'm': [1 - 1/df_soil.loc['10_4','npar'].item()],
              'L':[df_soil.loc['10_4','L'].item()]}
print(soil_props)


In [ ]:
input_dir = os.path.join(os.getcwd(),'hydrus_inputs','10_4_forcing_and_moisture.csv')
df_inputs = pd.read_csv(input_dir)
df_inputs
plt.plot(df_inputs['precip'].values[0:24*365])

In [ ]:
pond_max= 3.0
discretize = {"layers": [200], "dz": [1.0]} # one layer 200 cm 
bound_bot = 'free drainage' 
bound_top = 'atmospheric'
flux_top = (-df_inputs.loc[:,'precip'].values +  
            df_inputs.loc[:,'E0'].values)/ (60 * 10)# precipitation negative, evaporation positive mm/hour to cm/min coefficient

flux_bot = np.zeros(flux_top.shape) # dummy for free drainage
head_bot,head_top = np.zeros(flux_top.shape),np.ones(flux_top.shape) # dummies, top is atmospheric


In [ ]:
temp_time  = 60.0 # temporal resolution is hourly
sim_time = temp_time * flux_top.shape[0] # 5 year  > 365 * 24 * 60 minute
hyd_model = 'VGM-AE' # air entry genuchten model

In [ ]:
transpiration = df_inputs.loc[:,'T0'].values / (60 * 10) #mm/hour to cm/min coefficient 60 and 10
root_depth = 100.0 # cm surface to bottom 
root_model = 'feddes' # or feddes
root_dist = "uniform" # second option is "uniform"
root_params = response_feddes = {"p0": -10, "p0opt": -25, "p2h": -200, "p2l": -800,
                   "p3": -8000,"r2h":0.5 / 1440,"r2l":0.1/1440 }

In [ ]:
# new model 
model = InfiltrationModel(sim_time,temp_time,discretize,np.float32) #create the model!
hini = np.ones(model.nodes) * -100.0# initial pressure vector
model.set_soil_model(hyd_model,soil_props)
model.set_root_model(root_model,root_params,root_dist,root_depth)
model.set_boundary_conditions(bound_bot)
hout_x,sout_x = model.set_run_solver(hini,flux_top,transpiration,pond_max,time_interval=24)


In [ ]:
plt.plot(sout_x[:,-1])

In [ ]:
import time 

from model_build import InfiltrationModel
print("Starting simulation...",flush=True)
start_time = time.time()
model = InfiltrationModel(sim_time,temp_time,discretize) #create the model!
model.set_soil_model(hyd_model,soil_props,test=False) # test was for example3! always false
model.set_root_model(root_model,root_params,root_dist,root_depth)

hini = np.ones(model.nodes) * -100# initial pressure vector
model.set_boundary_conditions(hini,bound_top,bound_bot,pond_max)
hout,sout,sink_out = model.set_run_solver(hini,flux_top,flux_bot,head_top,head_bot,transpiration,time_interval=24)
# Calculate the difference
end_time = time.time()
execution_time = end_time - start_time
seconds = execution_time % 60
print(f'Simulation finished! Total run time: second {seconds:.2f}')


In [ ]:
print(hout.shape)

In [ ]:
import phydrus as ps
import os 
import time
# Folder for Hydrus files to be stored
print("Starting simulation...",flush=True)
start_time = time.time()
ws = "field_txson_example"
# Path to folder containing hydrus.exe 
exe = os.path.join('/Users/onursahin/Flux1Dpy/hydrus_source/source_code/source/hydrus')
# Description
desc = "TxSON real network example"
# Create model
ml = ps.Model(exe_name=exe, ws_name=ws, name="model", description=desc, 
              mass_units="mmol", time_unit="hours", length_unit="cm")

total_hours = 43824
print_time = np.arange(24,total_hours+1,24)
ml.add_time_info(tmax=total_hours, print_array=print_time, dt=0.5,dtmin=0.01,dtmax=1)
ml.add_waterflow(model=3, top_bc=3, bot_bc=4)

m = ml.get_empty_material_df(n=1)
#m.loc[[1]] = [[0.068, 0.38, 0.008, 1.09, 4.8, 0.5]]

m.loc[[1]] =[[df_soil.loc['10_4','theta_r'].item(),df_soil.loc['10_4','theta_s'].item(),
              df_soil.loc['10_4','alpha'].item(),df_soil.loc['10_4','npar'].item(),
              df_soil.loc['10_4','ksat'].item()/24,df_soil.loc['10_4','L'].item()
              ]]

ml.add_material(m)
elements = 200  # Disctretize soil column into n elements
depth = -200  # Depth of the soil column
ihead = -100  # Determine initial Pressure Head
# Create Profile
profile = ps.create_profile(bot=depth, dx=abs(depth / elements), h=ihead)
root_zone = profile['x'] >= -100
profile.loc[root_zone, 'Beta'] = 1.0
profile.loc[1, 'Beta'] = 0 # inactivate surface 
ml.add_profile(profile)  # Add the profile
ml.add_obs_nodes([0,-20])
tAtm = np.arange(1, total_hours  + 1)

bc = {"tAtm": tAtm, "Prec": df_inputs.loc[:,'precip'].values / 10, # cm/hour
      "rSoil": df_inputs.loc[:,'E0'].values / 10,  #cm/hour
      "rRoot": df_inputs.loc[:,'T0'].values / 10}
atm = pd.DataFrame(bc, index=tAtm)

ml.add_atmospheric_bc(atm, hcrits=0.0, hcrita=50000)
ml.add_root_uptake(model=0, crootmax=1, poptm=[-25], omegac=1)
ml.write_input()

ml.simulate()
end_time = time.time()
execution_time = end_time - start_time
seconds = execution_time % 60
print(f'Simulation finished! Total run time: second {seconds:.2f}')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import warnings

warnings.simplefilter(action='ignore', category=FutureWarning)

# Read the HYDRUS node information (profile snapshots)
hydrus_res = ml.read_obs_node()
fig,axs = plt.subplots(ncols=2,nrows=1,figsize = (10,5))
theta_hydrus = hydrus_res[21]['theta'].values
theta_custom = sout_x[:,-20]
print(theta_custom.shape,theta_hydrus.shape)
axs[0].plot(theta_hydrus,label='HYRUS at 20cm')
axs[0].plot(theta_custom,label='Custom Model at 20 cm')
#axs[0].plot(df_inputs.loc[np.arange(0,43824,24),'SWC_20'].values,label='Sensor Data')
axs[1].plot(theta_hydrus-theta_custom,label='Difference')
axs[1].set_ylim([-0.02,0.02])
#plt.plot(theta_custom,label='Custom Model')
axs[0].set_xlabel('First Hour of Days 2015-2019')
axs[0].set_ylabel('Soil Moisture Content')

axs[0].legend()
axs[1].legend()